# Academic Helpdesk Agent (multi-agent RAG) — Colab runner
Run the cells top to bottom. **Before starting:** `Runtime → Change runtime type → T4 GPU`.

Your project folder must already be in Google Drive at `MyDrive/helpdesk-agent/`
with your documents in `data/docs/` and your CSVs in `data/csv/` (see README).

Every new Colab session: re-run cells 1–7 (about 3–5 minutes), then use any cell after that.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

/bin/bash: line 1: nvidia-smi: command not found


## 2. Mount Google Drive and open the project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT = '/content/drive/MyDrive/helpdesk-agent'   # change if you put it elsewhere
os.environ['HELPDESK_DIR'] = PROJECT
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print('Documents found:', sorted(os.listdir('data/docs')))
print('CSV files found:', sorted(os.listdir('data/csv')))

Mounted at /content/drive
Documents found: ['.DS_Store', '.gitkeep', '01_BS_in_Engineering_Chemistry.pdf', '01_UG_EE_Curriculum-Final.pdf', '02_UG_Materials_Eng_Revised_Curr_Final.pdf', '03_Chemical_Engineering_UG_Booklet.pdf', '03_Concept_Note_B.S.Mathematics_Computing_2026.pdf', '04_UG_Curriculum_Structure_BSBE.pdf', '05_Revised-UG-Curriculum-CSE.pdf', '06_Revised-UG-Curriculum-AIDE.pdf', '07_UG_Curriculum_revised_Civil_and_Infrastructure_Engineering.pdf', '08_Concept_Note_BTech_ME.pdf', '09_Aerospace_Engg_merge-2026.pdf', '10_Curriculum-BTech_ECE.pdf', '3_Academic_Regulations_Final_03_09_2019.pdf', 'CourseStructure_BS_ MT.pdf', 'Curriculum-of-BS-program-in-Physics-639177222604928330.pdf', 'Curriculum-of-MSc-in-Physics-2026-639208557752166058.pdf', 'Curriculum3-638755770502183063.pdf', 'Programmes_2026-27.pdf', 'academic calender.pdf', 'attendance ammendment.pdf', 'examination_ammendment.pdf']
CSV files found: ['.DS_Store', 'AIDE - Sheet1 (1).csv', 'Aero courses - Sheet1.csv', 'BSBE 

## 3. Install Python packages

In [ ]:
!pip install -q -r requirements.txt

## 4. Install Ollama

In [ ]:
!sudo apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 4.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## 5. Start the Ollama server (8K context so tool schemas + passages fit)

In [ ]:
import subprocess, time, requests
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH='8192', OLLAMA_KEEP_ALIVE='30m')
ollama_proc = subprocess.Popen(['ollama', 'serve'], env=env,
                               stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get('http://localhost:11434'); print('Ollama is running'); break
    except Exception:
        time.sleep(1)

Ollama is running


## 6. Download the models (~5 GB, a few minutes)

In [ ]:
!ollama pull qwen3:8b
!ollama pull nomic-embed-text

## 6b. Build timetable.csv / courses.csv / faculty.csv from the department sheets
Re-run whenever you add or replace a sheet in `data/csv/`.

In [ ]:
!python prepare_csvs.py

Slot grid: 34 slots | faculty emails: 304 names
  + AIDE - Sheet1 (1).csv: 25 courses  (School of Artificial Intelligence and Data Science (AIDE))
  + Aero courses - Sheet1.csv: 3 courses  (Aerospace Engineering)
  + BSBE courses - Sheet1.csv: 33 courses  (Bioscience and Bioengineering)
  + CET courses - Sheet1.csv: 19 courses  (Centre for Education Technology)
  + CSE  - Sheet1.csv: 37 courses  (Computer Science and Engineering)
  + ECE  - Sheet1.csv: 20 courses  (Electronics and Communication Engineering)
  + EE courses - Sheet1.csv: 35 courses  (Electrical Engineering)
  + MT courses - Sheet1.csv: 43 courses  (Materials Engineering)
  + Math courses - Sheet1.csv: 43 courses  (Mathematics)
  + SME courses - Sheet1.csv: 42 courses  (School of Management and Entrepreneurship)
  + SOLA  - Sheet1.csv: 27 courses  (School of Liberal Arts (Humanities and Social Sciences))
  + chemical - Sheet1.csv: 30 courses  (Chemical Engineering)

Wrote 357 courses -> courses.csv, timetable.csv
faculty.

## 7. Build the vector store from your documents
Re-run this whenever you add or change documents.

In [ ]:
!python ingest.py

  + 01_BS_in_Engineering_Chemistry.pdf: 210 chunks
  + 01_UG_EE_Curriculum-Final.pdf: 50 chunks
  + 02_UG_Materials_Eng_Revised_Curr_Final.pdf: 251 chunks
  + 03_Chemical_Engineering_UG_Booklet.pdf: 210 chunks
  + 03_Concept_Note_B.S.Mathematics_Computing_2026.pdf: 148 chunks
  + 04_UG_Curriculum_Structure_BSBE.pdf: 26 chunks
  + 05_Revised-UG-Curriculum-CSE.pdf: 74 chunks
  + 06_Revised-UG-Curriculum-AIDE.pdf: 48 chunks


## 8. Sanity-check the tools directly (no LLM involved)

In [ ]:
import importlib, tools; importlib.reload(tools)
r = tools.search_documents('attendance requirement to appear in major exam')
for x in r.get('results', []):
    print(x['citation'], x['relevance'], '\n', x['text'][:300], '\n---')
print(r.get('message', ''))
print('400-level Humanities electives:',
      tools.list_courses(department='Humanities', level='400', category='Elective')['count'])

## 9. Single-agent version (base version)
`await` works directly in Colab cells.

In [ ]:
!pip install -q mcp

In [ ]:
from helpdesk import Helpdesk
single = Helpdesk('single')
r = await single.ask('What is the attendance requirement to sit in the Major exam?')
print(r['answer']); print(r['agents'], r['tools'], r['seconds'], 's')

## 10. Multi-agent version (orchestrator → policy / schedule / email agents)

In [ ]:
multi = Helpdesk('multi')
demo = [
    'What is the attendance requirement to sit in the Major exam?',
    'What is the slot for the course Autonomous Systems?',
    'How many 400 level electives have been floated by the Humanities department in this semester?',
    'How many Open Elective credits do I need to earn in order to get a B.Tech. Degree?',
    'Write an email to the course instructor of CSL7650 that I need leave for 2 days',
    'What are the available dining options on the campus?',
    'What is the best way to reach the campus from Jodhpur city center?',
]
for q in demo:
    try:
        r = await multi.ask(q)
    except Exception as e:
        print('Q:', q, '\nERROR:', e, '\n'); continue
    print('Q:', q); print('A:', r['answer'])
    print('   route:', ' -> '.join(r['agents']), '| tools:', r['tools'], '|', r['seconds'], 's\n')

## 11. Unanswered-questions log and email drafts

In [ ]:
import pandas as pd, glob
if os.path.exists('logs/unanswered_questions.csv'):
    display(pd.read_csv('logs/unanswered_questions.csv'))
for f in sorted(glob.glob('drafts/*.txt'))[-2:]:
    print(open(f).read(), '\n=====')

## 12. Evaluate (Qwen3 8B, multi-agent, 3 runs per question)
Takes roughly 15–30 min for 26 questions × 3 runs on a T4.

In [ ]:
!python evaluate.py --mode multi --runs 3

## 13. Swap the LLM: Llama 3.1 8B

In [ ]:
!ollama pull llama3.1:8b
!python evaluate.py --mode multi --runs 3 --model ollama_chat/llama3.1:8b

## 14. (Optional) Single-agent baseline for comparison

In [ ]:
!python evaluate.py --mode single --runs 3

## 15. Compare results

In [ ]:
comp = pd.read_csv('results/model_comparison.csv'); display(comp)
for f in sorted(glob.glob('results/per_question_*.csv')):
    df = pd.read_csv(f)
    changed = df[(df.distinct_answers > 1) | df.correctness_changed]
    print(f'\n{f}: {len(changed)} questions changed across runs')
    display(changed[['id', 'question', 'distinct_answers', 'correctness_changed', 'routing_changed']])

In [ ]:
# ============================================================
# ACADEMIC HELPDESK — INTERACTIVE COLAB UI
# Uses existing helpdesk.py, Google ADK agents, RAG and tools
# ============================================================

%pip install -q gradio

import asyncio
import importlib
import gradio as gr

import helpdesk
importlib.reload(helpdesk)
from helpdesk import Helpdesk

# Cache helpdesk instances so models/agents are not rebuilt
# for every message.
_HELPDESK_INSTANCES = {}


def get_helpdesk(mode, model):
    key = (mode, model or None)

    if key not in _HELPDESK_INSTANCES:
        _HELPDESK_INSTANCES[key] = Helpdesk(
            mode=mode,
            model=model or None
        )

    return _HELPDESK_INSTANCES[key]


def clear_chat():
    # A fresh chat gets a fresh ADK session on the next message.
    return [], "Ready for your next question.", "", None


async def respond(message, history, mode, model, session_state):
    message = (message or "").strip()
    history = list(history or [])

    if not message:
        return history, "Please enter a question.", "", session_state

    try:
        hd = get_helpdesk(mode, model)

        # Keep a session for follow-up questions.
        # Create a new session when the selected model/mode changes.
        session_key = (mode, model or None)

        if (
            not isinstance(session_state, dict)
            or session_state.get("key") != session_key
        ):
            session_id = await hd.new_session()
        else:
            session_id = session_state["session_id"]

        result = await hd.ask(message, session_id=session_id)

        answer = result.get("answer", "No answer was returned.")

        # Keep the existing helpdesk's answer and its citations intact.
        history.append({"role": "user", "content": message})
        history.append({"role": "assistant", "content": answer})

        agents = result.get("agents", [])
        tools = result.get("tools", [])
        elapsed = result.get("seconds", 0)

        # Display execution details separately from the answer.
        details = [
            f"**Mode:** {mode.title()}-agent",
            f"**Time:** {elapsed} s",
            f"**Agents:** {' → '.join(agents) if agents else 'Not reported'}",
            f"**Tools:** {', '.join(tools) if tools else 'None reported'}",
        ]

        status = "  \n".join(details)

        new_session_state = {
            "key": session_key,
            "session_id": session_id
        }

        return history, status, "", new_session_state

    except Exception as exc:
        # Keep the UI alive and show an actionable error.
        error = (
            f"**Request failed.** `{type(exc).__name__}: {exc}`\n\n"
            "Check that Ollama is running, the selected model is available, "
            "and the project files and ChromaDB index are accessible."
        )

        return history, error, message, session_state


CSS = """
.gradio-container {
    max-width: 1180px !important;
    margin: auto !important;
}
#app-header {
    padding: 25px 28px;
    border-radius: 18px;
    background: linear-gradient(120deg, #10294c, #1d4e89);
    color: white;
    margin-bottom: 16px;
}
#app-header h1 {
    color: white !important;
    margin-bottom: 8px !important;
}
#app-header p {
    color: #e0ebfa !important;
}
#chat-panel {
    border-radius: 16px;
}
#send-button {
    min-width: 110px;
}
#status-panel {
    border-radius: 12px;
}
"""

with gr.Blocks(
    title="Academic Helpdesk | IIT Jodhpur"
) as demo:

    gr.HTML("""
    <div id="app-header">
        <h1>🎓 Academic Helpdesk</h1>
        <p>
            Your AI-powered academic assistant · Powered by Google ADK,
            retrieval-augmented generation and local language models.
        </p>
    </div>
    """)

    session_state = gr.State(value=None)

    with gr.Row(equal_height=False):

        # ---------------- LEFT SIDEBAR ----------------
        with gr.Column(scale=1, min_width=245):
            gr.Markdown("### ⚙️ Configuration")

            mode = gr.Radio(
                choices=[
                    ("Multi-agent", "multi"),
                    ("Single-agent", "single")
                ],
                value="multi",
                label="Agent architecture"
            )

            model = gr.Dropdown(
                choices=[
                    ("Qwen3 8B · configured default", ""),
                    ("Llama 3.1 8B", "ollama_chat/llama3.1:8b")
                ],
                value="",
                label="Language model",
                info="The selected model must already be available in Ollama."
            )

            gr.Markdown("---")
            gr.Markdown("### 💡 Try asking")

            gr.Markdown(
                "- What is the attendance requirement for the Major exam?\n\n"
                "- What is the slot for Autonomous Systems?\n\n"
                "- How many Open Elective credits are required for a B.Tech. degree?\n\n"
                "- Draft an email requesting two days of leave for CSL7650.\n\n"
                "- What are the dining options on campus?"
            )

            gr.Markdown("---")

            gr.Markdown(
                "**Privacy & reliability**\n\n"
                "Answers depend on your indexed academic documents and "
                "CSV data. Email requests generate drafts only; emails "
                "are not sent."
            )

        # ---------------- MAIN CHAT ----------------
        with gr.Column(scale=3, min_width=400):
            gr.Markdown("### 💬 Ask your academic question")

            chatbot = gr.Chatbot(
                height=475,
                label="Conversation",
                show_label=False,
                placeholder=(
                    "Welcome to Academic Helpdesk!\n\n"
                    "Ask about academic regulations, curriculum, "
                    "course slots or instructor email drafts."
                )
            )

            with gr.Row():
                message = gr.Textbox(
                    placeholder="Type your academic question here...",
                    show_label=False,
                    scale=8,
                    lines=2,
                    max_lines=5,
                    autofocus=True
                )
                send = gr.Button(
                    "Send ➤",
                    variant="primary",
                    scale=1,
                    elem_id="send-button"
                )

            with gr.Row():
                clear = gr.Button("＋ New conversation")
                gr.Markdown(
                    "<div style=\'text-align:right;color:gray;"
                    "font-size:12px;padding-top:8px\'>"
                    "Academic knowledge base · Local LLM</div>"
                )

            with gr.Accordion("Execution details", open=False):
                status = gr.Markdown(
                    "Ready for your question.",
                    elem_id="status-panel"
                )

            gr.Markdown(
                "<small>AI-generated responses may be incorrect. "
                "Check the cited academic documents for authoritative rules. "
                "Unsupported questions should be referred for human review."
                "</small>"
            )

    # Send via either the Send button or Enter/submit in the textbox.
    inputs = [message, chatbot, mode, model, session_state]
    outputs = [chatbot, status, message, session_state]

    send.click(
        fn=respond,
        inputs=inputs,
        outputs=outputs
    )

    message.submit(
        fn=respond,
        inputs=inputs,
        outputs=outputs
    )

    clear.click(
        fn=clear_chat,
        inputs=[],
        outputs=[chatbot, status, message, session_state]
    )

    gr.Examples(
        examples=[
            ["What is the attendance requirement to sit in the Major exam?"],
            ["What is the slot for the course Autonomous Systems?"],
            ["How many 400 level electives have been floated by Humanities?"],
            ["How many Open Elective credits are required for a B.Tech. degree?"],
            ["Draft an email to the instructor of CSL7650 requesting two days of leave."],
            ["What are the available dining options on the campus?"],
        ],
        inputs=message,
        label="Suggested questions"
    )

demo.queue(default_concurrency_limit=1).launch(
    inline=True,
    share=False,
    debug=True,
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate"),
    css=CSS
)

In [ ]:
!pip install -q gradio
import gradio as gr
from helpdesk import Helpdesk

_hd = {}
async def respond(message, history):
    if 'hd' not in _hd:
        _hd['hd'] = Helpdesk('multi')
    r = await _hd['hd'].ask(message)          # no session_id = fresh session per question
    route = ' → '.join(r['agents'])
    tools = ', '.join(t for t in r['tools'] if t != 'transfer_to_agent')
    return f"{r['answer']}\n\n<sub>route: {route} | tools: {tools} | {r['seconds']} s</sub>"

gr.ChatInterface(respond, title="IIT Jodhpur Academic Helpdesk",
                 examples=["What is the slot for Autonomous Systems?",
                           "Draft an email to the instructor of CSL7650 requesting two days of leave",
                           "What course is Dr. Gaurav Harit teaching?",
                           "How many Open Elective credits are required for a B.Tech. degree?",
                           "What are the dining options on campus?"]).launch(share=True)